In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("IntrusionDetection") \
    .master("local[2]") \
    .getOrCreate()

df = spark.read.parquet("hdfs://namenode:8020/data/processed/features.parquet")
print(df.count())

2730540


In [2]:
from pyspark.sql.functions import row_number, from_unixtime, lit, to_timestamp
from pyspark.sql.window import Window

w = Window.orderBy(lit(1))
df = df.withColumn(
    "timestamp",
    to_timestamp(from_unixtime(lit(1421927414) + row_number().over(w)))
)

In [3]:
from pyspark.sql.functions import window, count, when, col

df.filter(col("attack_cat").isNotNull())\
  .groupBy(window("timestamp", "5 minutes"), "attack_cat")\
  .count()\
  .orderBy("window")\
  .show(10)

+--------------------+--------------+-----+
|              window|    attack_cat|count|
+--------------------+--------------+-----+
|{2015-01-26 23:20...|      Exploits|    7|
|{2015-01-26 23:20...|Reconnaissance|    2|
|{2015-01-26 23:20...|      Backdoor|    1|
|{2015-01-26 23:20...|           DoS|    7|
|{2015-01-26 23:20...|       Generic|    1|
|{2015-01-26 23:20...|       Fuzzers|    1|
|{2015-01-26 23:25...|      Exploits|   24|
|{2015-01-26 23:25...|       Fuzzers|    6|
|{2015-01-26 23:25...|           DoS|   15|
|{2015-01-26 23:25...|Reconnaissance|    5|
+--------------------+--------------+-----+
only showing top 10 rows



In [4]:
from pyspark.sql.functions import hour

df.filter(col("attack_cat").isNotNull())\
  .groupBy(hour("timestamp").alias("hour"))\
  .pivot("attack_cat")\
  .count()\
  .orderBy("hour")\
  .show()

+----+--------+--------+----+--------+-------+-------+--------------+---------+-----+
|hour|Analysis|Backdoor| DoS|Exploits|Fuzzers|Generic|Reconnaissance|Shellcode|Worms|
+----+--------+--------+----+--------+-------+-------+--------------+---------+-----+
|   0|     481|     505|2111|    3614|   1551|  10572|           696|       72|    6|
|   1|     223|     235|1246|    2869|   1374|   9512|           635|       62|   12|
|   2|     108|     122| 872|    2254|   1290|  10716|           622|       71|   10|
|   3|      24|      51| 356|    1519|   1484|  10640|           540|       70|    8|
|   4|      27|      29| 243|    1255|   1222|  10338|           508|       68|    6|
|   5|      10|      30| 227|    1245|   1046|  10115|           447|       52|    4|
|   6|      25|      27| 217|    1215|    724|   8832|           466|       51|    6|
|   7|      46|      36| 224|    1294|    855|   8472|           528|       58|    3|
|   8|     121|      42| 317|    1484|   1048|   7512|

In [5]:
from pyspark.sql.functions import avg, max, min, stddev

df.filter(col("attack_cat").isNotNull())\
  .groupBy("attack_cat")\
  .agg(
      avg("threat_score").alias("avg_threat"),
      max("threat_score").alias("max_threat"),
      stddev("threat_score").alias("threat_volatility")
  )\
  .orderBy("avg_threat", ascending=False)\
  .show()

+--------------+------------------+------------------+------------------+
|    attack_cat|        avg_threat|        max_threat| threat_volatility|
+--------------+------------------+------------------+------------------+
|      Exploits|2.6862615096596656|1002.3521230103261|25.729815668820052|
|       Fuzzers| 2.090706344947893|150.66483417634745| 4.566521386114914|
|           DoS|2.0089341285554934| 811.9810516047745|22.892024346954585|
|         Worms| 1.354295927611828|12.501940610658572|1.9995838436284052|
|     Shellcode|1.1641816155027316| 18.87041786734175|1.3901262943080603|
|Reconnaissance| 0.759216370199122|48.163934534291855|0.8362847325581221|
|       Generic|0.6121183824705186|250.50160108460193|1.5630097488969208|
|      Backdoor|0.5562468963489904|48.163934534291855|1.6288423324031513|
|      Analysis| 0.532823995466067|48.163934534291855|1.5308175266487098|
+--------------+------------------+------------------+------------------+



In [6]:
df.write.mode("overwrite")\
  .partitionBy("Label")\
  .parquet("hdfs://namenode:8020/data/partitioned/")

In [8]:
partitioned_df = spark.read.parquet("hdfs://namenode:8020/data/partitioned/")

partitioned_df.filter(col("Label") == 1)\
              .groupBy("attack_cat")\
              .count()\
              .explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[attack_cat#844], functions=[count(1)])
   +- Exchange hashpartitioning(attack_cat#844, 200), ENSURE_REQUIREMENTS, [plan_id=550]
      +- HashAggregate(keys=[attack_cat#844], functions=[partial_count(1)])
         +- Project [attack_cat#844]
            +- FileScan parquet [attack_cat#844,Label#860] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[hdfs://namenode:8020/data/partitioned], PartitionFilters: [isnotnull(Label#860), (Label#860 = 1)], PushedFilters: [], ReadSchema: struct<attack_cat:string>




In [9]:
# cache frequently used filtered dataframe
attack_df = partitioned_df.filter(col("Label") == 1).cache()
attack_df.count()  # trigger cache

# now run multiple queries on cached data
attack_df.groupBy("attack_cat").count().show()
attack_df.groupBy("proto").count().show()

+--------------+------+
|    attack_cat| count|
+--------------+------+
|         Worms|   174|
|     Shellcode|  1511|
|       Fuzzers| 24246|
|      Analysis|  2677|
|           DoS| 16353|
|Reconnaissance| 13987|
|      Backdoor|  2329|
|      Exploits| 44525|
|       Generic|215481|
+--------------+------+

+----------+-----+
|     proto|count|
+----------+-----+
|      cphb|  137|
|nsfnet-igp|  137|
|      larp|  137|
|       dgp|  137|
|       tcf|  137|
|     crudp|  137|
|       igp|  137|
|       nvp|  137|
|      vrrp|  137|
|   mfe-nsp|  137|
|        il|  137|
|       prm|  137|
|  wb-expak|  137|
|      micp|  137|
|      ospf| 3278|
|br-sat-mon|  137|
|      idrp|  137|
| kryptolan|  137|
|        ib|  137|
|sprite-rpc|  137|
+----------+-----+
only showing top 20 rows



In [10]:
attack_df.explain()  # shows InMemoryRelation in plan = cached

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- InMemoryTableScan [state#805, proto#806, dur#807, sbytes#808, dbytes#809, sttl#810, dttl#811, sloss#812, dloss#813, service#814, Sload#815, Dload#816, Spkts#817, Dpkts#818, swin#819, dwin#820, smeansz#821, dmeansz#822, trans_depth#823, res_bdy_len#824, Sjit#825, Djit#826, Sintpkt#827, Dintpkt#828, ... 32 more fields]
      +- InMemoryRelation [state#805, proto#806, dur#807, sbytes#808, dbytes#809, sttl#810, dttl#811, sloss#812, dloss#813, service#814, Sload#815, Dload#816, Spkts#817, Dpkts#818, swin#819, dwin#820, smeansz#821, dmeansz#822, trans_depth#823, res_bdy_len#824, Sjit#825, Djit#826, Sintpkt#827, Dintpkt#828, ... 32 more fields], StorageLevel(disk, memory, deserialized, 1 replicas)
            +- *(1) ColumnarToRow
               +- FileScan parquet [state#805,proto#806,dur#807,sbytes#808,dbytes#809,sttl#810,dttl#811,sloss#812,dloss#813,service#814,Sload#815,Dload#816,Spkts#817,Dpkts#818,swin#819,dwin#820,smeansz#821,d